# Named-entity recognition on WikiANN

This notebook compares a pretrained BERT token classifier with zero-shot entity extraction by Gemma. Predictions from both models are converted to word-level BIO labels and evaluated with entity-level F1.

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
from seqeval.metrics import precision_score, recall_score, f1_score
from nlp_portfolio.ner import pipeline_entities_to_dict, text_entities_to_bio, wikiann_gold_tags

MODEL_ID = 'dslim/bert-base-NER'
dataset = load_dataset('wikiann', 'en', split='validation[:64]')
label_names = dataset.features['ner_tags'].feature.names
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID)
ner = pipeline('token-classification', model=model, tokenizer=tokenizer, aggregation_strategy='simple',
               device=0 if torch.cuda.is_available() else -1)
print('Validation items:', len(dataset))

In [ ]:
gold, bert_predictions = [], []
for row in dataset:
    words = row['tokens']
    text = ' '.join(words)
    entities = pipeline_entities_to_dict(ner(text))
    gold.append(wikiann_gold_tags(row['ner_tags'], label_names))
    bert_predictions.append(text_entities_to_bio(words, entities))
print('BERT entity F1:', round(f1_score(gold, bert_predictions), 4))
print('BERT precision:', round(precision_score(gold, bert_predictions), 4))
print('BERT recall:', round(recall_score(gold, bert_predictions), 4))

## Gemma inference

Gemma produces a JSON object with entity spans. Invalid JSON is treated as an empty prediction. Gold tags are used only for evaluation.

In [ ]:
from transformers import AutoModelForCausalLM
from nlp_portfolio.ner import predict_chat_entities

GEMMA_ID = 'unsloth/gemma-3-1B-it'
chat_tokenizer = AutoTokenizer.from_pretrained(GEMMA_ID)
chat_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_ID, device_map='auto', dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
chat_predictions, raw_answers = [], []
for row in dataset:
    objects, raw = predict_chat_entities(' '.join(row['tokens']), chat_tokenizer, chat_model)
    chat_predictions.append(text_entities_to_bio(row['tokens'], objects))
    raw_answers.append(raw)
print('Gemma entity F1:', round(f1_score(gold, chat_predictions), 4))
print('Parsed sample:', raw_answers[0][:300])

BERT subword aggregation may produce imperfect spans. Exact matching of generated entities can also miss punctuation variants. The two models use the same WikiANN examples, and the reported names reflect the checkpoints actually loaded.